# Step 1: Load the Data and Split It
First, you need to load that massive Parquet file and use the split column (that your teammate brilliantly coded) to carve out your training, validation, and testing sets.

In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Load the data 
print("Loading data...")
df = pd.read_parquet("data/gkx2020_panel.parquet", engine="fastparquet") 

# 2. Recreate the Time-Series Splits using the 'date' column directly
print("Splitting data...")
train_end = pd.Timestamp("1974-12-31")
valid_end = pd.Timestamp("1986-12-31")
test_end  = pd.Timestamp("2021-12-31")

train_mask = df["date"] <= train_end
valid_mask = (df["date"] > train_end) & (df["date"] <= valid_end)
test_mask  = (df["date"] > valid_end) & (df["date"] <= test_end)

# 3. Define your features (X) and target (y)
# Notice we removed 'split' from cols_to_drop since it's not in the file
cols_to_drop = ['permno', 'date', 'exchcd', 'ret_excess']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

# 4. Apply the masks to create the datasets
print("Assigning X and y...")
X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, 'ret_excess']
X_valid, y_valid = df.loc[valid_mask, feature_cols], df.loc[valid_mask, 'ret_excess']
X_test,  y_test  = df.loc[test_mask,  feature_cols], df.loc[test_mask,  'ret_excess']

# Free up memory (Crucial for your MacBook!)
del df
gc.collect()

print("Data successfully loaded and split!")

Loading data...
Splitting data...
Assigning X and y...
Data successfully loaded and split!


# Step 2: Train the Random Forest
Because you have over 450,000 rows in the training set and 936 features, a standard Random Forest will take a very long time to train.

**Pro-Tip**: In the code below, max_depth=3 keeps the trees shallow so it runs quickly while you are just testing your code. Once you know it works, you can increase max_depth and tune it using the validation set.

# Step 3: Train the Gradient Boosted Trees (GBRT)
GBRT builds trees sequentially. It is more prone to overfitting than a Random Forest, which is why the learning_rate parameter is critical here.

# Step 4: Saving the Models (Run this right after training)
Add this code in a new cell immediately below where your rf_model.fit() and gbrt_model.fit() code runs.

# Step 5: Loading the Models (When you come back tomorrow)
When you close VS Code and come back later, you do not need to run the .fit() cells again.

Instead, you just need to run your first cell (to load and split the gkx2020_panel.parquet data), and then run this cell to instantly load your trained brains back into memory:

In [2]:
import joblib

print("Loading saved models from disk...")

# Load the Random Forest
rf_model = joblib.load('models/rf_model_gkx.pkl')

# Load the Gradient Boosted Trees
gbrt_model = joblib.load('models/gbrt_model_gkx.pkl')

print("Models loaded and ready for predictions!")

Loading saved models from disk...
Models loaded and ready for predictions!


# Step 6: The Ultimate Test (Calculate $R^2_{OOS}$)
This is the most important calculation for your part of the project. GKX (2020) does not use the standard scikit-learn $R^2$ score. In empirical asset pricing, the benchmark is predicting a zero return.You must calculate the Out-of-Sample R-squared exactly like this on your X_test data:

In [3]:
# 1. Fill missing values with 0 (The median value for cross-sectionally ranked data)
print("Filling missing values with 0...")
X_test = X_test.fillna(0)

# 2. Define the evaluation function
def calculate_oos_r2(y_true, y_pred):
    denominator = (y_true ** 2).sum()
    numerator = ((y_true - y_pred) ** 2).sum()
    r2_oos = 1 - (numerator / denominator)
    return r2_oos

# 3. Generate predictions
print("Generating predictions...")
rf_predictions = rf_model.predict(X_test)
gbrt_predictions = gbrt_model.predict(X_test)

# 4. Calculate and print the results
rf_r2 = calculate_oos_r2(y_test, rf_predictions)
gbrt_r2 = calculate_oos_r2(y_test, gbrt_predictions)

print(f"Random Forest R2_OOS: {rf_r2 * 100:.3f}%")
print(f"Gradient Boosting R2_OOS: {gbrt_r2 * 100:.3f}%")

Filling missing values with 0...
Generating predictions...
Random Forest R2_OOS: -1.115%
Gradient Boosting R2_OOS: -11.825%


# Step 7: Save the Predictions (Run this once)
Right after you generate rf_predictions and gbrt_predictions in your notebook, run this cell to bundle them into a clean dataset and save it to your disk.

# Step 8: Load the Predictions (When you come back)
Tomorrow, when you open your laptop to work on Phase 3 (the Portfolio Sorting and economic evaluation), you don't need to load your models or your massive 11 GB dataset.

You just need to run this single cell:

In [4]:
import pandas as pd

print("Loading saved predictions...")

# Load the predictions directly from your hard drive
results_df = pd.read_parquet("data/model_predictions.parquet", engine="fastparquet")

# Now you can instantly access your numbers!
y_test_loaded = results_df['actual_return']
rf_preds_loaded = results_df['rf_predicted']
gbrt_preds_loaded = results_df['gbrt_predicted']

print(f"Loaded {len(results_df):,} rows of predictions.")

Loading saved predictions...
Loaded 2,138,514 rows of predictions.


# Sharpe ratio step

In [7]:
import pandas as pd
import numpy as np

print("Loading test dates and Market Equity...")

# 1. Load date AND Market Equity ('me') for value-weighting
all_data_subset = pd.read_parquet("data/gkx2020_panel.parquet", columns=["date", "me"], engine="fastparquet")

valid_end = pd.Timestamp("1986-12-31")
test_end  = pd.Timestamp("2021-12-31")
test_mask = (all_data_subset["date"] > valid_end) & (all_data_subset["date"] <= test_end)

# Isolate the test dates and market cap
test_subset = all_data_subset.loc[test_mask].reset_index(drop=True)

print("Loading predictions...")
results_df = pd.read_parquet("data/model_predictions.parquet", engine="fastparquet")

# Attach dates and market cap to your predictions
results_df['date'] = test_subset['date']
results_df['me'] = test_subset['me']

# 2. Define the Value-Weighted Portfolio Logic
def calculate_vw_return(df_month, prediction_col):
    if len(df_month) < 10:
        return np.nan
        
    try:
        df_month['decile'] = pd.qcut(df_month[prediction_col], 10, labels=False, duplicates='drop')
    except:
        return np.nan
    
    # Isolate the Long bucket (top 10%) and Short bucket (bottom 10%)
    long_leg = df_month[df_month['decile'] == df_month['decile'].max()]
    short_leg = df_month[df_month['decile'] == df_month['decile'].min()]
    
    # Calculate Value-Weighted Returns: (Stock Return * (Stock ME / Total Bucket ME))
    long_vw_ret = (long_leg['actual_return'] * (long_leg['me'] / long_leg['me'].sum())).sum()
    short_vw_ret = (short_leg['actual_return'] * (short_leg['me'] / short_leg['me'].sum())).sum()
    
    return long_vw_ret - short_vw_ret

# 3. Calculate month-by-month Value-Weighted returns
print("Calculating realistic Value-Weighted returns...")
monthly_vw_rf = results_df.groupby('date').apply(calculate_vw_return, prediction_col='rf_predicted').dropna()
monthly_vw_gbrt = results_df.groupby('date').apply(calculate_vw_return, prediction_col='gbrt_predicted').dropna()

# 4. Evaluate Economic Performance
def print_performance(monthly_returns, model_name):
    ann_return = monthly_returns.mean() * 12
    ann_volatility = monthly_returns.std() * np.sqrt(12)
    sharpe_ratio = ann_return / ann_volatility if ann_volatility != 0 else 0
    
    print(f"\n--- {model_name} VALUE-WEIGHTED Portfolio ---")
    print(f"Annualized Return: {ann_return * 100:.2f}%")
    print(f"Annualized Volatility: {ann_volatility * 100:.2f}%")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

print_performance(monthly_vw_rf, "Random Forest")
print_performance(monthly_vw_gbrt, "Gradient Boosted Trees")

Loading test dates and Market Equity...
Loading predictions...
Calculating realistic Value-Weighted returns...

--- Random Forest VALUE-WEIGHTED Portfolio ---
Annualized Return: 25.78%
Annualized Volatility: 22.34%
Sharpe Ratio: 1.15

--- Gradient Boosted Trees VALUE-WEIGHTED Portfolio ---
Annualized Return: 213.56%
Annualized Volatility: 37.21%
Sharpe Ratio: 5.74


# Roadmap Step

In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from tqdm.auto import tqdm
import os

# 1. Load the data
print("Loading data...")
df = pd.read_parquet("data/gkx2020_panel.parquet", engine="fastparquet")

# Ensure date is a datetime object so we can extract the year
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

# Define features
cols_to_drop = ['permno', 'date', 'exchcd', 'ret_excess', 'split', 'year']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

# 2. Setup the models
rf_model = RandomForestRegressor(n_estimators=300, max_depth=3, max_features='sqrt', random_state=42, n_jobs=-1)
gbrt_model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=3, random_state=42)

# 3. Create a place to store our out-of-sample predictions
all_predictions = []
os.makedirs("data/yearly_preds", exist_ok=True) 

# 4. THE TWO-YEAR TEST LOOP (1987 and 1988 only)
years_to_predict = range(1987, 1989) 

print("Starting 2-Year Test Run...")
for target_year in tqdm(years_to_predict, desc="Years Processed"):
    
    # Expanding Window: Train on everything BEFORE the target year
    train_mask = df['year'] < target_year
    test_mask = df['year'] == target_year
    
    if test_mask.sum() == 0:
        continue
        
    X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, 'ret_excess']
    X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, 'ret_excess']
    
    # Fill NAs with 0 (median)
    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)
    
    # Train the models
    rf_model.fit(X_train, y_train)
    gbrt_model.fit(X_train, y_train)
    
    # Predict the target year
    rf_preds = rf_model.predict(X_test)
    gbrt_preds = gbrt_model.predict(X_test)
    
    # Store the results for this year
    yearly_results = pd.DataFrame({
        'date': df.loc[test_mask, 'date'],
        'actual_return': y_test,
        'rf_predicted': rf_preds,
        'gbrt_predicted': gbrt_preds
    })
    
    all_predictions.append(yearly_results)
    
    # Save test backups
    yearly_results.to_parquet(f"data/yearly_preds/TEST_preds_{target_year}.parquet", engine="fastparquet")
    
    # Free up RAM instantly
    del X_train, y_train, X_test, y_test, yearly_results
    gc.collect()

# 5. Stitch the 2 years together
final_predictions_df = pd.concat(all_predictions, ignore_index=True)

# Save the test master file
final_predictions_df.to_parquet("data/TEST_recursive_predictions.parquet", engine="fastparquet")
print("2-Year test complete and saved!")

# 6. Calculate the test R2_OOS
def calculate_oos_r2(y_true, y_pred):
    denominator = (y_true ** 2).sum()
    numerator = ((y_true - y_pred) ** 2).sum()
    return 1 - (numerator / denominator)

true_rf_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df['rf_predicted'])
true_gbrt_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df['gbrt_predicted'])

print(f"TEST RUN - Random Forest R2_OOS (2 Years): {true_rf_r2 * 100:.3f}%")
print(f"TEST RUN - Gradient Boosting R2_OOS (2 Years): {true_gbrt_r2 * 100:.3f}%")

Loading data...
Starting 2-Year Test Run...


Years Processed:   0%|          | 0/2 [00:00<?, ?it/s]

2-Year test complete and saved!
TEST RUN - Random Forest R2_OOS (2 Years): -0.243%
TEST RUN - Gradient Boosting R2_OOS (2 Years): 14.407%
